# MODULE 5: Offline Policy Learning

### NOVEL CONTRIBUTION:
This module implements an Acuity-Conditioned Decision Transformer (AC-DT) with
multi-objective return conditioning and safety-aware action masking.

### KEY INNOVATIONS:
1. Acuity-conditioned architecture (conditions on SOFA score trajectory)
2. Multi-objective return-to-go (NO scalarization - vector R̂)
3. Safety-aware action masking (prevents extreme dosing)
4. Continuous-time modeling (handles irregular sampling)
5. Uncertainty-aware planning (uses reward uncertainty from Module 4)

### MATHEMATICAL FRAMEWORK:

Standard Decision Transformer:

    π(a_t | s_{0:t}, a_{0:t-1}, R̂_t) where R̂_t = Σ_{τ=t}^T γ^{τ-t} r_τ
    
Our Acuity-Conditioned Multi-Objective DT:

    π(a_t | s_{0:t}, a_{0:t-1}, R̂_t, c_t, Δt_{0:t})

where:

    R̂_t ∈ ℝ^k (multi-objective returns)

    c_t = acuity(s_t) ∈ [0, 1] (SOFA-based severity)

    Δt_{0:t} (irregular time intervals)
    
Safety Constraint:

    A_safe(s_t, c_t) = {a : Q_min(s_t, a, c_t) ≥ θ_safe}

    π(a_t) is masked to A_safe
    
LITERATURE FOUNDATION:

[1] Chen et al. "Decision Transformer: RL via Sequence Modeling." NeurIPS 2021 </br>
[2] Kumar et al. "Conservative Q-Learning for Offline RL." NeurIPS 2020 </br>
[3] Janner et al. "Planning with Diffusion for Flexible Behavior Synthesis." ICML 2022 </br>
[4] Emmons et al. "RvS: What is Essential for Offline RL via SSL?" ICLR 2022 </br>
[5] Farebrother et al. "Stop Regressing: Training Value Functions via Classification." 2024 </br>
[6] Komorowski et al. "The AI Clinician learns optimal treatment strategies." Nat Med 2018 </br>

COMPARISON TO ALTERNATIVES:
- CQL: Requires explicit value function (unstable in healthcare, sparse rewards)
- IQL: Still value-based, doesn't model long-term dependencies well
- Standard DT: No safety, no acuity awareness, scalar returns
- Our AC-DT: Addresses all these issues

In [ ]:
class ContinuousTimeEncoding(nn.Module):
    """
    Encodes irregular time intervals for continuous-time modeling.
    
    PROBLEM: ICU measurements are irregularly sampled (not every hour)
    SOLUTION: Fourier time embeddings that handle arbitrary Δt
    
    Based on: "Attention is All You Need" positional encoding +
              "SeFT: Learning Irregular Time Series" (Gong et al. 2020)
    """
    def __init__(self, d_model: int):
        super().__init__()
        self.d_model = d_model
        
        # Learnable frequency scaling
        self.omega = nn.Parameter(torch.randn(d_model // 2))
        
    def forward(self, delta_t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            delta_t: [B, T] - time intervals in hours
        Returns:
            time_embed: [B, T, d_model]
        """
        # Fourier features
        # shape: [B, T, d_model//2]
        phase = delta_t.unsqueeze(-1) * self.omega.unsqueeze(0).unsqueeze(0)
        
        # sin/cos encoding
        time_embed = torch.cat([
            torch.sin(phase),
            torch.cos(phase)
        ], dim=-1)
        
        return time_embed

In [ ]:
# ACUITY SCORING

class AcuityScorer(nn.Module):
    """
    Computes patient acuity score from state.
    
    CLINICAL DEFINITION:
    Acuity = severity of illness, used to stratify treatment intensity
    
    IMPLEMENTATION:
    - Primary: SOFA score (validated organ dysfunction measure)
    - Secondary: Vital sign instability, vasopressor dependency
    
    Output: c ∈ [0, 1] where 0=stable, 1=critical
    """
    def __init__(self, d_state: int = 70):
        super().__init__()
        
        # Indices in state vector 
        self.sofa_idx = 21  
        
        # Learnable weights for multi-signal acuity
        self.acuity_net = nn.Sequential(
            nn.Linear(d_state, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()  # Output in [0, 1]
        )
        
    def forward(self, states: torch.Tensor) -> torch.Tensor:
        """
        Args:
            states: [B, T, D_state]
        Returns:
            acuity: [B, T] in [0, 1]
        """
        return self.acuity_net(states).squeeze(-1)

In [ ]:
if __name__ == "__main__":

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # ========================== 1. LOAD DATA FROM MODULE 1 ==========================
    print("\n[1] Loading trajectories from Module 1...")
    extractor = CohortExtractor()
    cohort = extractor.extract_cohort()
    
    feature_extractor = FeatureExtractor(FeatureConfig())
    action_discretizer = ActionDiscretizer(ActionBins(), data_dir='../../../data/')
    feature_imputer = fit_imputer_on_cohort(cohort, feature_extractor)
    
    traj_builder = TrajectoryBuilder(feature_extractor, action_discretizer, feature_imputer)
    trajectories = traj_builder.build_dataset(cohort)
    
    splitter = DatasetSplitter()
    splits = splitter.split(trajectories)
    
    print(f"   → Train: {len(splits['train'])} | Val: {len(splits['val'])} | Test: {len(splits['test'])} trajectories")

    # ========================== 2. LOAD PRE-TRAINED MODELS ==========================
    print("\n[2] Loading pre-trained models...")

    # State Encoder (Module 2)
    state_encoder = HistoryAwareStateEncoder(d_state=76).to(device)
    se_ckpt = torch.load('state_encoder.pt', map_location=device)
    state_encoder.load_state_dict(se_ckpt.get('encoder', se_ckpt))
    state_encoder.eval()
    print("   ✅ State Encoder loaded")

    # Reward Model (Module 4)
    reward_model = MultiObjectiveRewardModel(d_state=76).to(device)
    rm_ckpt = torch.load('multiobjective_reward_model.pt', map_location=device)
    reward_model.load_state_dict(rm_ckpt)
    reward_model.eval()
    print("   ✅ Multi-objective Reward Model loaded")

    # ========================== 3. INITIALIZE POLICY ==========================
    print("\n[3] Initializing Acuity-Conditioned Decision Transformer...")
    policy = AcuityConditionedDecisionTransformer(
        d_state=76,
        n_actions=25,
        n_objectives=4,
        d_model=256,
        n_layers=6,
        n_heads=8,
        dropout=0.1
    ).to(device)

    # ========================== 4. CREATE DATALOADERS ==========================
    print("\n[4] Creating offline policy dataloaders...")
    train_loader, val_loader = create_policy_dataloaders(
        train_trajectories=splits['train'],
        val_trajectories=splits['val'],
        reward_model=reward_model,
        batch_size=16          
    )

    # ========================== 5. TRAIN ==========================
    print("\n[5] Starting training...")
    trainer = DecisionTransformerTrainer(policy, device=device)
    
    trained_policy = trainer.train(
        train_loader=train_loader,
        val_loader=val_loader,
        n_epochs=80,        
        lambda_cql=0.3,    
        save_path='acuity_conditioned_dt.pt'
    )

Using device: cpu

[1] Loading trajectories from Module 1...
No DB config provided. Using CSV mode.


INFO:__main__:ActionDiscretizer initialized with data for 75 icustays.


✅ Extracted 34 sepsis trajectories.

🔧 Fitting FeatureImputer on raw cohort data...


  Learned population statistics from 2108 training samples
✅ Imputer fitted successfully on 2,108 hourly records
🔧 Building trajectories...


Processing patients: 100%|██████████| 34/34 [02:18<00:00,  4.08s/it]


...


Epoch 11/80
  Train: BC Loss=46662256.2500 | CQL Loss=2.0472 | Action Acc=0.940
  Val:   BC Loss=184848448.0000 | Action Acc=0.815 | Avg Acuity=0.167



Epoch 12/80
  Train: BC Loss=43567684.1250 | CQL Loss=1.9744 | Action Acc=0.940
  Val:   BC Loss=184848448.0000 | Action Acc=0.815 | Avg Acuity=0.165



Epoch 13/80
  Train: BC Loss=95623466.0000 | CQL Loss=1.9449 | Action Acc=0.940
  Val:   BC Loss=184848448.0000 | Action Acc=0.815 | Avg Acuity=0.162



Epoch 14/80
  Train: BC Loss=71877453.0000 | CQL Loss=1.7078 | Action Acc=0.940
  Val:   BC Loss=184848448.0000 | Action Acc=0.815 | Avg Acuity=0.159


KeyboardInterrupt: 

In [ ]:
# SAFETY-AWARE ACTION MASKING

class SafetyConstraint(nn.Module):
    """
    APPROACH:
    - Learn conservative Q-function Q_min(s, a, c)
    - Mask actions where Q_min < threshold
    - Prevents: excessive fluids (→ pulmonary edema), extreme vasopressors
    
    Based on: Kumar et al. "Conservative Q-Learning" (NeurIPS 2020)
    """
    def __init__(self,
                 d_state: int = 70,
                 d_acuity: int = 1,
                 n_actions: int = 25,
                 d_hidden: int = 256):
        super().__init__()
        
        self.n_actions = n_actions
        
        # Conservative Q-network
        self.q_net = nn.Sequential(
            nn.Linear(d_state + d_acuity, d_hidden),
            nn.ReLU(),
            nn.Linear(d_hidden, d_hidden),
            nn.ReLU(),
            nn.Linear(d_hidden, n_actions)
        )
        
        # Safety threshold (learned)
        self.log_threshold = nn.Parameter(torch.tensor(0.0))
        
    def forward(self, states: torch.Tensor, acuity: torch.Tensor) -> torch.Tensor:
        """
        Compute action safety mask.
        
        Args:
            states: [B, T, D_state]
            acuity: [B, T]
        Returns:
            mask: [B, T, n_actions] - 1 if safe, 0 if unsafe
        """
        B, T, D = states.shape
        
        # Append acuity to state
        sa = torch.cat([states, acuity.unsqueeze(-1)], dim=-1)
        
        # Q-values
        q_values = self.q_net(sa)  # [B, T, n_actions]
        
        # Safety mask: Q > threshold
        threshold = torch.exp(self.log_threshold)
        mask = (q_values > threshold).float()
        
        # Ensure at least one action is available (fallback to safest)
        n_safe = mask.sum(dim=-1, keepdim=True)
        fallback = (n_safe == 0).float()
        
        if fallback.sum() > 0:
            # If no safe actions, allow the one with highest Q
            best_actions = q_values.argmax(dim=-1, keepdim=True)
            fallback_mask = torch.zeros_like(mask)
            fallback_mask.scatter_(-1, best_actions, 1.0)
            mask = mask + fallback * fallback_mask
        
        return mask
    
    def update(self,
               states: torch.Tensor,
               actions: torch.Tensor,
               returns: torch.Tensor,
               acuity: torch.Tensor,
               optimizer: torch.optim.Optimizer) -> float:
        """
        Conservative Q-learning update for safety constraint.
        Expects flattened or properly shaped inputs.
        """
        # Flatten batch and time dimensions for Q-learning update
        if states.dim() == 3:   # [B, T, D] or [B, 1, D]
            states = states.view(-1, states.shape[-1])
        if actions.dim() == 3:  # [B, 1, T] or similar
            actions = actions.view(-1)
        if returns.dim() == 3:
            returns = returns.view(-1)
        if acuity.dim() == 3:
            acuity = acuity.view(-1)
        
        B = states.shape[0] 
        
        # Compute Q-values for all actions
        sa = torch.cat([states, acuity.unsqueeze(-1)], dim=-1)
        q_all = self.q_net(sa)                    # [B, n_actions]
        
        # Q-values for the taken actions
        q_taken = q_all.gather(-1, actions.unsqueeze(-1)).squeeze(-1)   # [B]
        
        # Bellman error (MSE)
        bellman_loss = F.mse_loss(q_taken, returns)
        
        # Conservative penalty (LogSumExp)
        logsumexp = torch.logsumexp(q_all, dim=-1)   # [B]
        conservative_penalty = logsumexp.mean() - q_taken.mean()
        
        # Total loss
        alpha = 1.0
        loss = bellman_loss + alpha * conservative_penalty
        
        # Optimize
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_net.parameters(), 1.0)
        optimizer.step()
        
        return loss.item()

In [ ]:
# ACUITY-CONDITIONED DECISION TRANSFORMER

class AcuityConditionedDecisionTransformer(nn.Module):
    """
    Architecture:
        Input: (s_t, a_{t-1}, R̂_t, c_t, Δt_t) for t=0..T
        Embed each modality
        GPT-style causal transformer
        Predict: a_t ~ π(· | history, R̂_t, c_t, safety_mask)
    
    Key Differences from Standard DT:
    - Standard DT: π(a | s, R̂) with scalar R̂
    - AC-DT: π(a | s, R̂, c, Δt, mask) with vector R̂ ∈ ℝ^k
    """
    def __init__(self,
                 d_state: int = 70,
                 n_actions: int = 25,
                 n_objectives: int = 4,
                 d_model: int = 256,
                 n_layers: int = 6,
                 n_heads: int = 8,
                 dropout: float = 0.1,
                 max_ep_len: int = 168):
        super().__init__()
        
        self.d_state = d_state
        self.n_actions = n_actions
        self.n_objectives = n_objectives
        self.d_model = d_model
        self.max_ep_len = max_ep_len
        
        # Embeddings for each modality
        self.state_embed = nn.Linear(d_state, d_model)
        self.action_embed = nn.Embedding(n_actions, d_model)
        self.return_embed = nn.Sequential(
            nn.Linear(n_objectives, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model) # extra projection
        )

        self.acuity_embed = nn.Linear(1, d_model)
        
        # Continuous time encoding
        self.time_encoder = ContinuousTimeEncoding(d_model)
        
        # Position embeddings (relative to episode start)
        self.pos_embed = nn.Parameter(torch.zeros(1, max_ep_len * 3, d_model))
        
        # Layer norm for embeddings
        self.embed_ln = nn.LayerNorm(d_model)
        
        # GPT-style transformer
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerDecoder(
            decoder_layer,
            num_layers=n_layers
        )
        
        # Prediction heads

        self.action_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.LayerNorm(d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, n_actions)
        )
        
        # Acuity scorer
        self.acuity_scorer = AcuityScorer(d_state)
        
        # Safety constraint
        self.safety = SafetyConstraint(d_state, 1, n_actions)
        
        print(f"✅ AcuityConditionedDecisionTransformer initialized:")
        print(f"   State: {d_state}, Actions: {n_actions}, Objectives: {n_objectives}")
        print(f"   Model dim: {d_model}, Layers: {n_layers}, Heads: {n_heads}")
        print(f"   Innovations: Acuity conditioning, Multi-objective, Continuous-time, Safety")
        
    def forward(self,
                states: torch.Tensor,
                actions: torch.Tensor,
                returns_to_go: torch.Tensor,
                timesteps: torch.Tensor,
                delta_t: torch.Tensor,
                attention_mask: Optional[torch.Tensor] = None,
                return_acuity: bool = False) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        Forward pass through AC-DT.
        
        Args:
            states: [B, T, D_state]
            actions: [B, T] - previous actions (use dummy for t=0)
            returns_to_go: [B, T, k] - multi-objective returns
            timesteps: [B, T] - episode timesteps
            delta_t: [B, T] - time since last observation (hours)
            attention_mask: [B, T] - mask for variable length
            return_acuity: whether to return acuity scores
            
        Returns:
            action_logits: [B, T, n_actions]
            acuity: [B, T] (optional)
        """
        B, T, _ = states.shape
        
        # Compute acuity
        acuity = self.acuity_scorer(states)  # [B, T]
        
        # Embed each modality
        state_embeds = self.state_embed(states)
        action_embeds = self.action_embed(actions)
        return_embeds = self.return_embed(returns_to_go)
        acuity_embeds = self.acuity_embed(acuity.unsqueeze(-1))
        
        # Time embeddings (continuous)
        time_embeds = self.time_encoder(delta_t)
        
        # Interleave: (R̂_0, s_0, a_0, R̂_1, s_1, a_1, ...)
        # This is the standard DT pattern
        sequence = torch.stack([
            return_embeds + time_embeds,
            state_embeds + acuity_embeds + time_embeds,
            action_embeds + time_embeds
        ], dim=2).reshape(B, 3 * T, self.d_model)
        
        # Add positional embeddings
        if 3 * T <= self.max_ep_len * 3:
            pos_embed = self.pos_embed[:, :3*T, :]
        else:
            # Interpolate if sequence too long
            pos_embed = F.interpolate(
                self.pos_embed.transpose(1, 2),
                size=3*T,
                mode='linear'
            ).transpose(1, 2)
        
        sequence = sequence + pos_embed
        sequence = self.embed_ln(sequence)
        
        # Causal mask (prevent looking at future)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(
            3 * T,
            device=states.device
        )
        
        # Transformer
        hidden = self.transformer(
            sequence,
            sequence,  # Self-attention (decoder-only)
            tgt_mask=causal_mask
        )
        
        # Extract state positions (every 3rd token starting from index 1)
        # Pattern: [R̂, s, a, R̂, s, a, ...]
        #           0   1  2  3   4  5
        state_hidden = hidden[:, 1::3, :]  # [B, T, d_model]
        
        # Predict actions
        action_logits = self.action_head(state_hidden)  # [B, T, n_actions]
        
        # Apply safety mask
        safety_mask = self.safety(states, acuity)  # [B, T, n_actions]
        
        # Mask out unsafe actions (set logits to -inf)
        action_logits = action_logits + (1 - safety_mask) * (-1e9)
        
        if return_acuity:
            return action_logits, acuity
        else:
            return action_logits
    
    def get_action(self,
                   states: torch.Tensor,
                   actions: torch.Tensor,
                   returns_to_go: torch.Tensor,
                   timesteps: torch.Tensor,
                   delta_t: torch.Tensor,
                   temperature: float = 1.0,
                   deterministic: bool = False) -> torch.Tensor:
        """
        Sample action at last timestep (for autoregressive generation).
        
        Args:
            states: [B, T, D_state]
            actions: [B, T]
            returns_to_go: [B, T, k]
            timesteps: [B, T]
            delta_t: [B, T]
            temperature: sampling temperature
            deterministic: if True, take argmax
            
        Returns:
            action: [B] - sampled action
        """
        with torch.no_grad():
            action_logits = self.forward(
                states, actions, returns_to_go,
                timesteps, delta_t
            )
            
            # Take last timestep
            logits = action_logits[:, -1, :] / temperature
            
            if deterministic:
                action = logits.argmax(dim=-1)
            else:
                probs = F.softmax(logits, dim=-1)
                action = torch.multinomial(probs, num_samples=1).squeeze(-1)
        
        return action


In [ ]:
# TRAINING PIPELINE

class DecisionTransformerTrainer:
    """
    Trains the Acuity-Conditioned Decision Transformer.
    
    Training Objectives:
    1. Behavioral cloning (maximize likelihood of observed actions)
    2. Conservative Q-learning (for safety constraint)
    
    Loss:
        L = L_BC + λ L_CQL
    where:
        L_BC = -E[log π(a_t | s_{≤t}, R̂_t, c_t)]
        L_CQL = conservative penalty on Q-function
    """
    def __init__(self,
                 model: AcuityConditionedDecisionTransformer,
                 device: str = 'cuda'):
        self.model = model
        self.device = device
        
        # Separate optimizers for policy and Q-function
        self.policy_optimizer = torch.optim.AdamW(
            [p for n, p in model.named_parameters() if 'safety' not in n],
            lr=1e-4,
            weight_decay=0.01
        )
        
        self.q_optimizer = torch.optim.AdamW(
            model.safety.parameters(),
            lr=3e-4,
            weight_decay=0.01
        )
        
        print("✅ DecisionTransformerTrainer initialized")
        
    def train(self,
              train_loader: torch.utils.data.DataLoader,
              val_loader: torch.utils.data.DataLoader,
              n_epochs: int = 100,
              lambda_cql: float = 0.1,
              save_path: str = 'policy.pt'):
        """
        Train policy with behavior cloning + conservative Q-learning.
        
        Args:
            train_loader: DataLoader with trajectories
            val_loader: Validation DataLoader
            n_epochs: Number of epochs
            lambda_cql: Weight for CQL loss
            save_path: Where to save best model
        """
        
        scheduler_policy = torch.optim.lr_scheduler.CosineAnnealingLR(
            self.policy_optimizer, T_max=n_epochs
        )
        scheduler_q = torch.optim.lr_scheduler.CosineAnnealingLR(
            self.q_optimizer, T_max=n_epochs
        )
        
        best_val_acc = 0
        
        print(f"\n🏋️ Training AC-DT for {n_epochs} epochs...")
        
        for epoch in range(n_epochs):
            train_metrics = self._train_epoch(
                train_loader,
                lambda_cql
            )
            
            val_metrics = self._validate(val_loader)
            
            scheduler_policy.step()
            scheduler_q.step()
            
            # Logging
            print(f"\nEpoch {epoch+1}/{n_epochs}")
            print(f"  Train: BC Loss={train_metrics['bc_loss']:.4f} | "
                  f"CQL Loss={train_metrics['cql_loss']:.4f} | "
                  f"Action Acc={train_metrics['action_acc']:.3f}")
            print(f"  Val:   BC Loss={val_metrics['bc_loss']:.4f} | "
                  f"Action Acc={val_metrics['action_acc']:.3f} | "
                  f"Avg Acuity={val_metrics['avg_acuity']:.3f}")
            
            # Save best model
            if val_metrics['action_acc'] > best_val_acc:
                best_val_acc = val_metrics['action_acc']
                torch.save({
                    'model': self.model.state_dict(),
                    'epoch': epoch,
                    'val_acc': best_val_acc
                }, save_path)
                print(f"  ✅ Saved (val_acc: {best_val_acc:.3f})")
        
        # Load best
        checkpoint = torch.load(save_path)
        self.model.load_state_dict(checkpoint['model'])
        
        print(f"\n✅ Training complete! Best val acc: {best_val_acc:.3f}")
        
        return self.model
    
    def _train_epoch(self, loader, lambda_cql: float):
        self.model.train()
        
        total_bc_loss = 0.0
        total_cql_loss = 0.0
        total_correct = 0
        total_actions = 0
        
        for batch in tqdm(loader, desc="Training", leave=False):
            # Move to device
            states = batch['states'].to(self.device)
            actions = batch['actions'].to(self.device)
            returns_to_go = batch['returns_to_go'].to(self.device)
            timesteps = batch['timesteps'].to(self.device)
            delta_t = batch['delta_t'].to(self.device)
            mask = batch['mask'].to(self.device)
            
            # ==================== POLICY FORWARD (Behavior Cloning) ====================
            self.policy_optimizer.zero_grad()
            
            action_logits, acuity = self.model(
                states, actions, returns_to_go, timesteps, delta_t, return_acuity=True
            )
            
            # BC loss: predict next action
            action_targets = actions[:, 1:]           # shift by 1
            action_preds = action_logits[:, :-1]      # remove last prediction
            
            # Masked loss
            valid_mask = mask[:, 1:] == 1
            action_targets_flat = action_targets[valid_mask]
            action_preds_flat = action_preds[valid_mask]
            
            # Stable loss: log_softmax + NLLLoss
            log_probs = F.log_softmax(action_preds_flat, dim=-1)
            bc_loss = F.nll_loss(log_probs, action_targets_flat)
            
            # Backward for policy only
            bc_loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.policy_optimizer.step()
            
            # ==================== SAFETY Q-UPDATE (Separate) ====================
            with torch.no_grad():
                # Flatten for Q-update
                states_flat = states.view(-1, states.shape[-1]).detach()
                acuity_flat = acuity.view(-1).detach()
                actions_flat = actions.view(-1).detach()
                returns_flat = returns_to_go[:, :, 0].view(-1).detach()   # survival objective
                
            # Update conservative Q-function
            cql_loss_val = self.model.safety.update(
                states_flat,
                actions_flat,
                returns_flat,
                acuity_flat,
                self.q_optimizer
            )
            
            # ==================== METRICS ====================
            total_bc_loss += bc_loss.item()
            total_cql_loss += cql_loss_val
            
            with torch.no_grad():
                preds = action_preds_flat.argmax(dim=-1)
                total_correct += (preds == action_targets_flat).sum().item()
                total_actions += action_targets_flat.numel()
        
        metrics = {
            'bc_loss': total_bc_loss / len(loader),
            'cql_loss': total_cql_loss / len(loader),
            'action_acc': total_correct / total_actions if total_actions > 0 else 0.0
        }
        
        return metrics
    
    def _validate(self, loader):
        self.model.eval()
        
        total_bc_loss = 0.0
        total_correct = 0
        total_actions = 0
        total_acuity = 0.0
        n_samples = 0
        
        with torch.no_grad():
            for batch in loader:
                states = batch['states'].to(self.device)
                actions = batch['actions'].to(self.device)
                returns_to_go = batch['returns_to_go'].to(self.device)
                timesteps = batch['timesteps'].to(self.device)
                delta_t = batch['delta_t'].to(self.device)
                mask = batch['mask'].to(self.device)
                
                action_logits, acuity = self.model(
                    states, actions, returns_to_go, timesteps, delta_t, return_acuity=True
                )
                
                 # BC loss using stable log_softmax + NLL
                action_targets = actions[:, 1:]
                action_preds = action_logits[:, :-1]
                
                valid_mask = mask[:, 1:] == 1
                action_targets_flat = action_targets[valid_mask]
                action_preds_flat = action_preds[valid_mask]
                
                log_probs = F.log_softmax(action_preds_flat, dim=-1)
                bc_loss = F.nll_loss(log_probs, action_targets_flat)
                
                total_bc_loss += bc_loss.item()
                
                preds = action_preds_flat.argmax(dim=-1)
                total_correct += (preds == action_targets_flat).sum().item()
                total_actions += action_targets_flat.numel()
                
                total_acuity += acuity[mask == 1].mean().item()
                n_samples += 1
        
        metrics = {
            'bc_loss': total_bc_loss / len(loader),
            'action_acc': total_correct / total_actions if total_actions > 0 else 0.0,
            'avg_acuity': total_acuity / n_samples if n_samples > 0 else 0.0
        }
        
        return metrics

In [ ]:
# DATASET AND DATALOADER

class OfflinePolicyDataset(torch.utils.data.Dataset):
    """
    Dataset for training Decision Transformer on offline trajectories.
    
    Each sample is a full trajectory with:
    - states: [T, D_state]
    - actions: [T]
    - rewards: [T, k] - multi-objective
    - returns_to_go: [T, k] - cumulative future rewards
    - timesteps: [T] - relative to episode start
    - delta_t: [T] - time since last observation
    """
    def __init__(self,
                 trajectories: List,
                 reward_model: nn.Module,
                 n_objectives: int = 4,
                 rtg_scale: float = 0.001,
                 max_rtg: float = 0.1):
        self.trajectories = trajectories
        self.reward_model = reward_model
        self.n_objectives = n_objectives
        self.rtg_scale = rtg_scale
        self.max_rtg = max_rtg
        
        print("📊 Pre-computing return-to-go statistics...")
        all_rtg = []
        
        for traj in tqdm(trajectories[:min(100, len(trajectories))], desc="Sampling RTG"):
            rewards = self._get_rewards(traj)
            rtg = self._compute_returns_to_go(rewards)
            all_rtg.append(rtg)
        
        all_rtg = torch.cat(all_rtg, dim=0)
        
        self.rtg_mean = all_rtg.mean(dim=0)   # [k]
        self.rtg_std = all_rtg.std(dim=0) + 1e-6
        
        print(f"   RTG mean per objective: {self.rtg_mean.numpy()}")
        print(f"   RTG std per objective:  {self.rtg_std.numpy()}")
        print(f"   Applied clipping at ±{max_rtg}")
        
    def _get_rewards(self, traj):
        states = torch.from_numpy(traj.states).float().unsqueeze(0)
        
        if len(traj.actions.shape) == 2:
            a_idx = traj.actions[:, 0] * 5 + traj.actions[:, 1]
        else:
            a_idx = traj.actions
        actions = torch.from_numpy(a_idx).long().unsqueeze(0)
        
        with torch.no_grad():
            self.reward_model.eval()
            actions_onehot = F.one_hot(actions, num_classes=25).float()
            rewards = self.reward_model(states, actions_onehot).squeeze(0)
        
        return rewards
    
    def _compute_returns_to_go(self, rewards):
        T, k = rewards.shape
        returns_to_go = torch.zeros_like(rewards)
        returns_to_go[-1] = rewards[-1]
        for t in range(T-2, -1, -1):
            returns_to_go[t] = rewards[t] + returns_to_go[t+1]
        return returns_to_go
        
    def __len__(self):
        return len(self.trajectories)
    
    def __getitem__(self, idx):
        traj = self.trajectories[idx]
        T = traj.length
        
        # States
        states = torch.from_numpy(traj.states).float()
        
        # Actions as indices (0-24)
        if len(traj.actions.shape) == 2:
            actions = (traj.actions[:, 0] * 5 + traj.actions[:, 1]).astype(np.int64)
        else:
            actions = traj.actions.astype(np.int64)
        actions = torch.from_numpy(actions).long()
        
        # Use learned reward model from Module 4
        with torch.no_grad():
            self.reward_model.eval()
            
            # One-hot actions for reward model
            actions_onehot = F.one_hot(actions, num_classes=25).float()
            
            # Get vector-valued rewards [T, 4]
            rewards = self.reward_model(
                states.unsqueeze(0),
                actions_onehot.unsqueeze(0)
            ).squeeze(0)
        
        # Compute returns-to-go (cumulative future rewards)
        returns_to_go = torch.zeros_like(rewards)
        returns_to_go[-1] = rewards[-1]
        for t in range(T-2, -1, -1):
            returns_to_go[t] = rewards[t] + returns_to_go[t+1]
        
        # Robust normalization + clipping
        returns_to_go = (returns_to_go - self.rtg_mean) / self.rtg_std
        returns_to_go = torch.clamp(returns_to_go, -self.max_rtg, self.max_rtg)
        returns_to_go = returns_to_go * self.rtg_scale
        
        # Timesteps and delta_t
        timesteps = torch.arange(T, dtype=torch.long)
        delta_t = torch.ones(T, dtype=torch.float32)
        
        return {
            'states': states,
            'actions': actions,
            'rewards': rewards,
            'returns_to_go': returns_to_go,
            'timesteps': timesteps,
            'delta_t': delta_t,
            'length': T
        }

In [ ]:
def collate_trajectories(batch):
    """
    Collate variable-length trajectories with padding.
    """
    if len(batch) == 0:
        raise ValueError("Empty batch received")
    
    max_len = max(item['length'] for item in batch)
    B = len(batch)
    
    # Get dimensions from first item
    D_state = batch[0]['states'].shape[1]
    k = batch[0]['rewards'].shape[1]
    
    # Initialize padded tensors
    states_padded = torch.zeros(B, max_len, D_state, dtype=torch.float32)
    actions_padded = torch.zeros(B, max_len, dtype=torch.long)
    rewards_padded = torch.zeros(B, max_len, k, dtype=torch.float32)
    returns_to_go_padded = torch.zeros(B, max_len, k, dtype=torch.float32)
    timesteps_padded = torch.zeros(B, max_len, dtype=torch.long)
    delta_t_padded = torch.zeros(B, max_len, dtype=torch.float32)
    mask_padded = torch.zeros(B, max_len, dtype=torch.float32)
    
    for i, item in enumerate(batch):
        T = item['length']
        states_padded[i, :T] = item['states']
        actions_padded[i, :T] = item['actions']
        rewards_padded[i, :T] = item['rewards']
        returns_to_go_padded[i, :T] = item['returns_to_go']
        timesteps_padded[i, :T] = item['timesteps']
        delta_t_padded[i, :T] = item['delta_t']
        mask_padded[i, :T] = 1.0
    
    return {
        'states': states_padded,
        'actions': actions_padded,
        'rewards': rewards_padded,
        'returns_to_go': returns_to_go_padded,
        'timesteps': timesteps_padded,
        'delta_t': delta_t_padded,
        'mask': mask_padded
    }

In [ ]:
def create_policy_dataloaders(train_trajectories: List,
                                val_trajectories: List,
                                reward_model: nn.Module,
                                batch_size: int = 32):
    """
    Create dataloaders for policy training.
    """
    train_dataset = OfflinePolicyDataset(train_trajectories, reward_model)
    val_dataset = OfflinePolicyDataset(val_trajectories, reward_model)
    
    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_trajectories,
        num_workers=0
    )
    
    val_loader = torch.utils.data.DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_trajectories,
        num_workers=0
    )
    
    return train_loader, val_loader

In [ ]:
if __name__ == "__main__":

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # ========================== 1. LOAD DATA FROM MODULE 1 ==========================
    print("\n[1] Loading trajectories from Module 1...")
    extractor = CohortExtractor()
    cohort = extractor.extract_cohort()
    
    feature_extractor = FeatureExtractor(FeatureConfig())
    action_discretizer = ActionDiscretizer(ActionBins(), data_dir='../../../data/')
    feature_imputer = fit_imputer_on_cohort(cohort, feature_extractor)
    
    traj_builder = TrajectoryBuilder(feature_extractor, action_discretizer, feature_imputer)
    trajectories = traj_builder.build_dataset(cohort)
    
    splitter = DatasetSplitter()
    splits = splitter.split(trajectories)
    
    print(f"   → Train: {len(splits['train'])} | Val: {len(splits['val'])} | Test: {len(splits['test'])} trajectories")

    # ========================== 2. LOAD PRE-TRAINED MODELS ==========================
    print("\n[2] Loading pre-trained models...")

    # State Encoder (Module 2)
    state_encoder = HistoryAwareStateEncoder(d_state=76).to(device)
    se_ckpt = torch.load('state_encoder.pt', map_location=device)
    state_encoder.load_state_dict(se_ckpt.get('encoder', se_ckpt))
    state_encoder.eval()
    print("   ✅ State Encoder loaded")

    # Reward Model (Module 4)
    reward_model = MultiObjectiveRewardModel(d_state=76).to(device)
    rm_ckpt = torch.load('multiobjective_reward_model.pt', map_location=device)
    reward_model.load_state_dict(rm_ckpt)
    reward_model.eval()
    print("   ✅ Multi-objective Reward Model loaded")

    # ========================== 3. INITIALIZE POLICY ==========================
    print("\n[3] Initializing Acuity-Conditioned Decision Transformer...")
    policy = AcuityConditionedDecisionTransformer(
        d_state=76,
        n_actions=25,
        n_objectives=4,
        d_model=256,
        n_layers=6,
        n_heads=8,
        dropout=0.1
    ).to(device)

    # ========================== 4. CREATE DATALOADERS ==========================
    print("\n[4] Creating offline policy dataloaders...")
    train_loader, val_loader = create_policy_dataloaders(
        train_trajectories=splits['train'],
        val_trajectories=splits['val'],
        reward_model=reward_model,
        batch_size=16          
    )

    # ========================== 5. TRAIN ==========================
    print("\n[5] Starting training...")
    trainer = DecisionTransformerTrainer(policy, device=device)
    
    trained_policy = trainer.train(
        train_loader=train_loader,
        val_loader=val_loader,
        n_epochs=80,        
        lambda_cql=0.3,    
        save_path='acuity_conditioned_dt.pt'
    )

Using device: cpu

[1] Loading trajectories from Module 1...
No DB config provided. Using CSV mode.


INFO:__main__:ActionDiscretizer initialized with data for 75 icustays.


✅ Extracted 34 sepsis trajectories.

🔧 Fitting FeatureImputer on raw cohort data...


  Learned population statistics from 2108 training samples
✅ Imputer fitted successfully on 2,108 hourly records
🔧 Building trajectories...


Processing patients: 100%|██████████| 34/34 [02:18<00:00,  4.08s/it]


...


Epoch 11/80
  Train: BC Loss=46662256.2500 | CQL Loss=2.0472 | Action Acc=0.940
  Val:   BC Loss=184848448.0000 | Action Acc=0.815 | Avg Acuity=0.167



Epoch 12/80
  Train: BC Loss=43567684.1250 | CQL Loss=1.9744 | Action Acc=0.940
  Val:   BC Loss=184848448.0000 | Action Acc=0.815 | Avg Acuity=0.165



Epoch 13/80
  Train: BC Loss=95623466.0000 | CQL Loss=1.9449 | Action Acc=0.940
  Val:   BC Loss=184848448.0000 | Action Acc=0.815 | Avg Acuity=0.162



Epoch 14/80
  Train: BC Loss=71877453.0000 | CQL Loss=1.7078 | Action Acc=0.940
  Val:   BC Loss=184848448.0000 | Action Acc=0.815 | Avg Acuity=0.159


KeyboardInterrupt: 